# 04 — Recall & 510(k) Integration

Phase 3 pipeline: clean recalls and 510(k) clearances, map recalls to classification dimension.

**Outputs:**
- `clean_recall.parquet` — recall/enforcement data with mapping quality tiers
- `clean_510k.parquet` — 510(k) clearances with standardized applicant names

In [ ]:
import sys

sys.path.insert(0, "..")

from src.cleaning.clearances import clean_clearances
from src.cleaning.recalls import clean_recalls
from src.mapping.recall_product_code import map_recall_to_classification
from src.qa.checks import (
    check_coverage,
    check_null_rate,
    check_row_count,
    check_uniqueness,
    check_volume_shift,
    run_checks,
)

## 1. Clean Recalls

In [ ]:
recalls = clean_recalls()
print(f"Shape: {recalls.shape}")
print("\nYear distribution:")
print(recalls["recall_year"].value_counts().sort_index().to_string())
print("\nRecall class distribution:")
print(recalls["recall_class"].value_counts().to_string())
recalls.head()

## 2. Map Recalls to Classification

In [ ]:
mapped_recalls = map_recall_to_classification(recall_df=recalls)

tier_counts = mapped_recalls["mapping_quality"].value_counts()
tier_pcts = mapped_recalls["mapping_quality"].value_counts(normalize=True)
print("Mapping tier breakdown:")
for tier in tier_counts.index:
    print(f"  {tier}: {tier_counts[tier]:,} ({tier_pcts[tier]:.1%})")

core_pct = mapped_recalls["include_in_core_dashboard"].mean()
print(f"\nCore dashboard inclusion rate: {core_pct:.1%}")

## 3. Clean 510(k) Clearances

In [ ]:
clearances = clean_clearances()
print(f"Shape: {clearances.shape}")
print("\nYear distribution:")
print(clearances["decision_year"].value_counts().sort_index().to_string())
print("\nDecision code distribution:")
print(clearances["decision_code"].value_counts().to_string())
clearances.head()

## 4. QA Summary

In [ ]:
import pandas as pd

# Recall QA
recall_checks = [
    check_row_count(mapped_recalls, "clean_recall", min_rows=100),
    check_uniqueness(mapped_recalls, ["recall_number"], "recall_number_pk"),
    check_null_rate(mapped_recalls, "recall_initiation_date"),
    check_null_rate(mapped_recalls, "product_code", max_rate=0.30),
    check_null_rate(mapped_recalls, "recall_class", max_rate=0.05),
    check_volume_shift(mapped_recalls, "recall_year"),
]

# Mapping coverage
tier_1_2 = mapped_recalls["mapping_quality"].isin(["exact_product_code_match", "high_confidence_text_match"])
tier_1_2_pct = tier_1_2.mean() if len(mapped_recalls) > 0 else 0.0

# 510(k) QA
dim_pc = pd.read_parquet("../data/clean/dim_product_code.parquet")
valid_codes = set(dim_pc["product_code"].dropna())

clearance_checks = [
    check_row_count(clearances, "clean_510k", min_rows=100),
    check_uniqueness(clearances, ["k_number"], "k_number_pk"),
    check_null_rate(clearances, "decision_date"),
    check_null_rate(clearances, "product_code", max_rate=0.05),
    check_volume_shift(clearances, "decision_year"),
    check_coverage(clearances, "product_code", reference_values=valid_codes),
]

all_checks = recall_checks + clearance_checks
summary = run_checks(all_checks)
summary.style.map(
    lambda v: (
        "background-color: #d4edda"
        if v == "pass"
        else ("background-color: #fff3cd" if v == "warn" else ("background-color: #f8d7da" if v == "fail" else ""))
    ),
    subset=["status"],
)

## 5. Go/No-Go Gate

In [ ]:
# Gate 1: Recall mapping coverage (tiers 1+2 >= 60%)
print(f"Recall mapping tiers 1+2: {tier_1_2_pct:.1%} (target >= 60%)")
gate_recall = tier_1_2_pct >= 0.60
print(f"  -> {'PASS' if gate_recall else 'FAIL'}")

# Gate 2: 510(k) product code match rate (>= 90%)
pc_match_rate = clearances["product_code"].isin(valid_codes).mean() if len(clearances) > 0 else 0.0
print(f"\n510(k) product code match rate: {pc_match_rate:.1%} (target >= 90%)")
gate_510k = pc_match_rate >= 0.90
print(f"  -> {'PASS' if gate_510k else 'FAIL'}")

# Overall
any_fail = any(r.status == "fail" for r in all_checks)
print(f"\nQA checks: {'ALL PASS' if not any_fail else 'HAS FAILURES'}")
print(f"\n{'=' * 40}")
all_gates = gate_recall and gate_510k and not any_fail
print(f"OVERALL: {'GO — proceed to Phase 4' if all_gates else 'NO-GO — review issues above'}")